# Experiment C Tuning Results

Analyze the small W&B-tracked Experiment C global-model tuning sweep and identify the checkpoint to use for the later personalization run.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import wandb

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError("WANDB_API_KEY is not set. Export it before reading W&B runs.")

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "ssmcgm")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY")
api = wandb.Api()
if WANDB_ENTITY:
    PROJECT_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}"
else:
    PROJECT_PATH = f"{api.viewer().entity}/{WANDB_PROJECT}"

PROJECT_PATH


In [ ]:
runs = list(api.runs(PROJECT_PATH, filters={"config.experiment_name": "exp_C_tuning"}))
rows = []
for run in runs:
    cfg = dict(run.config)
    summary = dict(run.summary)
    row = {
        "run_id": run.id,
        "run_name": run.name,
        "state": run.state,
        "created_at": run.created_at,
        "learning_rate": cfg.get("learning_rate", summary.get("learning_rate")),
        "dropout": cfg.get("dropout", summary.get("dropout")),
        "weight_decay": cfg.get("weight_decay", summary.get("weight_decay")),
        "batch_size_per_gpu": cfg.get("batch_size_per_gpu", summary.get("batch_size_per_gpu")),
        "global_batch_size": cfg.get("global_batch_size", summary.get("global_batch_size")),
        "max_val_windows": cfg.get("max_val_windows", summary.get("max_val_windows")),
        "val_loss": summary.get("val_loss"),
        "val_MAE": summary.get("val_MAE"),
        "val_RMSE": summary.get("val_RMSE"),
        "val_MAPE": summary.get("val_MAPE"),
        "val_SMAPE": summary.get("val_SMAPE"),
        "best_epoch": summary.get("best_epoch"),
        "best_val_loss": summary.get("best_val_loss"),
        "best_checkpoint_gcs_path": summary.get("best_checkpoint_gcs_path"),
        "gcs_output_path": summary.get("gcs_output_path"),
        "runtime_hours": summary.get("runtime_hours"),
        "exit_code": summary.get("exit_code"),
        "had_nan_warning": summary.get("had_nan_warning"),
        "had_unknown_classes_warning": summary.get("had_unknown_classes_warning"),
        "had_bus_error": summary.get("had_bus_error"),
        "had_nccl_error": summary.get("had_nccl_error"),
        "had_oom_error": summary.get("had_oom_error"),
    }
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No exp_C_tuning W&B runs found.")

df = df.sort_values(["val_MAE", "val_RMSE", "best_val_loss"], na_position="last").reset_index(drop=True)
display(df)


In [ ]:
best_by_mae = df.dropna(subset=["val_MAE"]).sort_values("val_MAE").head(1)
best_by_rmse = df.dropna(subset=["val_RMSE"]).sort_values("val_RMSE").head(1)

print("Best by val_MAE")
display(best_by_mae)
print("Best by val_RMSE")
display(best_by_rmse)


In [ ]:
history_frames = []
for run in runs:
    hist = run.history(keys=["epoch", "train_loss_epoch", "val_loss"], pandas=True)
    if hist.empty:
        continue
    hist["run_name"] = run.name
    history_frames.append(hist)

if history_frames:
    histories = pd.concat(history_frames, ignore_index=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)
    for run_name, sdf in histories.groupby("run_name"):
        if "train_loss_epoch" in sdf:
            axes[0].plot(sdf["epoch"], sdf["train_loss_epoch"], label=run_name, alpha=0.7)
        if "val_loss" in sdf:
            axes[1].plot(sdf["epoch"], sdf["val_loss"], label=run_name, alpha=0.7)
    axes[0].set_title("Train Loss")
    axes[1].set_title("Validation Loss")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.grid(True, alpha=0.3)
    axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
else:
    print("No W&B history curves found yet.")


In [ ]:
plot_df = df.copy()
metrics = ["val_MAE", "val_RMSE", "best_val_loss"]
hparams = ["learning_rate", "dropout", "weight_decay", "batch_size_per_gpu", "max_val_windows"]

fig, axes = plt.subplots(len(metrics), len(hparams), figsize=(18, 10))
for i, metric in enumerate(metrics):
    for j, hp in enumerate(hparams):
        ax = axes[i, j]
        sdf = plot_df.dropna(subset=[metric, hp])
        ax.scatter(sdf[hp], sdf[metric], alpha=0.8)
        ax.set_xlabel(hp)
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
candidate_metric = "val_MAE" if df["val_MAE"].notna().any() else "val_RMSE"
recommended = df.dropna(subset=[candidate_metric]).sort_values(candidate_metric).head(1)
if recommended.empty:
    recommended = df.dropna(subset=["best_val_loss"]).sort_values("best_val_loss").head(1)

print("Recommended checkpoint for personalization")
display(recommended[["run_name", "val_MAE", "val_RMSE", "best_val_loss", "best_checkpoint_gcs_path"]])
if not recommended.empty:
    print(recommended.iloc[0]["best_checkpoint_gcs_path"])
